In [ ]:
'''# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session'''

## MLP (Feature-Only)

In this experiment,  a 2-layer Multi-Layer Perceptron (MLP) is trained using only node features, completely ignoring the graph structure.

To ensure a fair comparison, the same hyperparameters (learning rate and hidden dimension) are used as in the GAT model.

In [1]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.4 MB/s eta 0:00:00


In [2]:
import torch
import torch_geometric

print(torch.__version__)
print(torch_geometric.__version__)

2.10.0+cu128
2.7.0


In [3]:
from torch_geometric.datasets import Amazon
dataset=Amazon(root='/kaggle/working/Amazon',name='Computers')
data=dataset[0]
print(data)


Processing...


Data(x=[13752, 767], edge_index=[2, 491722], y=[13752])


Done!


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_idx = torch.load("/kaggle/input/datasets/mrxn251/gnn-splits/train_idx.pt")
val_idx = torch.load("/kaggle/input/datasets/mrxn251/gnn-splits/val_idx.pt")
test_idx = torch.load("/kaggle/input/datasets/mrxn251/gnn-splits/test_idx.pt")
train_idx = train_idx.to(device)
val_idx = val_idx.to(device)
test_idx = test_idx.to(device)


In [6]:
data = data.to(device)

print(len(train_idx), len(val_idx), len(test_idx))



9626 2063 2063


### MLP model architecture 

In [7]:
import torch
import torch.nn.functional as F

class MLP(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        
        self.lin1 = torch.nn.Linear(in_channels, hidden_channels)
        self.lin2 = torch.nn.Linear(hidden_channels, out_channels)

    def forward(self, x):
        x = self.lin1(x)
        x = F.relu(x)
        x = self.lin2(x)
        return x

In [23]:
def train_mlp(model, loader, optimizer, device,x,y):
    model.train()
    total_loss = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        out = model(x[batch])  

        loss = F.cross_entropy(out,y[batch])
        

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [45]:
@torch.no_grad()
def evaluate_mlp(model, loader, device,x,y):
    model.eval()
    correct = 0
    total = 0

    for batch in loader:
        batch = batch.to(device)

        out = model(x[batch])   
        pred = out.argmax(dim=1)

        correct += (pred == y[batch]).sum().item()
        total += batch.size(0)

    return correct / total

In [27]:
from sklearn.metrics import f1_score

@torch.no_grad()
def test_mlp(model, loader, device,x,y):
    model.eval()
    
    all_preds = []
    all_labels = []

    for batch in loader:
        batch = batch.to(device)

        out = model(x[batch])
        pred = out.argmax(dim=1)

        all_preds.append(pred.cpu())
        all_labels.append(y[batch].cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    acc = (all_preds == all_labels).mean()
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    micro_f1 = f1_score(all_labels, all_preds, average='micro')

    return acc, macro_f1, micro_f1

In [28]:
#using same values of hyperparameters hidden_dim and lr  as used in GAT 
best_config=best_config= {'hidden_dim': 8, 'lr': 0.001}

In [33]:
import random
import numpy as np
def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [47]:
from torch.utils.data import DataLoader
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score

def run_experiment_mlp(config, seed=0, run_test=False):
    set_seed(seed)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    epochs = 50 if not run_test else 100

    #  Model
    model = MLP(
        in_channels=data.num_features,
        hidden_channels=config["hidden_dim"],
        out_channels=dataset.num_classes
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["lr"],
        weight_decay=5e-4
    )

    #  DataLoaders
    train_loader = DataLoader(train_idx, batch_size=128, shuffle=True)
    val_loader = DataLoader(val_idx, batch_size=256)
    test_loader = DataLoader(test_idx, batch_size=256)

    x = data.x.to(device)
    y = data.y.to(device)

    best_val = 0
    patience = 10
    counter = 0
    save_model = run_test

    # Training loop
    for epoch in range(1, epochs + 1):
        model.train()
        loss = train_mlp(model, train_loader, optimizer, device,x,y)
        val_acc = evaluate_mlp(model, val_loader, device,x,y)

        
        # if run_test:
        #     print(f"Epoch {epoch:03d} | Loss: {total_loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_val:
            best_val = val_acc
            counter = 0

            if save_model:
                torch.save(model.state_dict(), f"/kaggle/working/best_modelMLP_{seed}.pt")
        else:
            counter += 1

        if counter >= patience:
            break

    if not run_test:
        return best_val

    # Load best model
    model.load_state_dict(
        torch.load(f"/kaggle/working/best_modelMLP_{seed}.pt", map_location=device)
    )

    #  Test
    acc, macro_f1, micro_f1 = test_mlp(model, test_loader, device,x,y)

    return acc, macro_f1, micro_f1

In [48]:
acc_list = []
micro_f1_list = []
macro_f1_list=[]
for seed in range(10):
    acc, macro_f1, micro_f1 = run_experiment_mlp(best_config, seed=seed, run_test=True)
    acc_list.append(acc)
    macro_f1_list.append(macro_f1)
    micro_f1_list.append(micro_f1)

    print(f"Seed {seed}: Acc={acc:.4f}, macro_F1={macro_f1:.4f},micro_F1={micro_f1:.4f}")


Seed 0: Acc=0.8444, macro_F1=0.7894,micro_F1=0.8444
Seed 1: Acc=0.8434, macro_F1=0.7802,micro_F1=0.8434
Seed 2: Acc=0.8381, macro_F1=0.7912,micro_F1=0.8381
Seed 3: Acc=0.8429, macro_F1=0.7842,micro_F1=0.8429
Seed 4: Acc=0.8468, macro_F1=0.7979,micro_F1=0.8468
Seed 5: Acc=0.8376, macro_F1=0.8029,micro_F1=0.8376
Seed 6: Acc=0.8396, macro_F1=0.7856,micro_F1=0.8396
Seed 7: Acc=0.8492, macro_F1=0.7988,micro_F1=0.8492
Seed 8: Acc=0.8512, macro_F1=0.8008,micro_F1=0.8512
Seed 9: Acc=0.8492, macro_F1=0.8020,micro_F1=0.8492


In [49]:
import numpy as np

acc_mean = np.mean(acc_list)
acc_std = np.std(acc_list)

macro_f1_mean = np.mean(macro_f1_list)
macro_f1_std = np.std(macro_f1_list)

micro_f1_mean = np.mean(micro_f1_list)
micro_f1_std = np.std(micro_f1_list)

print(f"\nFinal Results:")
print(f"Accuracy: {acc_mean:.4f} ± {acc_std:.4f}")
print(f"Macro F1: {macro_f1_mean:.4f} ± {macro_f1_std:.4f}")
print(f"Micro F1: {micro_f1_mean:.4f} ± {micro_f1_std:.4f}")


Final Results:
Accuracy: 0.8443 ± 0.0046
Macro F1: 0.7933 ± 0.0078
Micro F1: 0.8443 ± 0.0046
